# Internal Fourier Audit: Where Is the Structure Hiding?

## Motivation

A reviewer raised a valid concern: the existing Fourier analysis checks **outputs**
(target encoder codes, context encoder per-class means, predictor outputs) but not
**internals** (embedding weights, hidden activations, predictor weights, individual neurons).

Nanda et al. (2023) specifically found Fourier structure in:
1. **Embedding weights** $W_E$ — tokens embedded on a circle
2. **Neuron-logit map** $W_L$ — sinusoidal functions of key frequencies
3. **MLP activations** projected onto sinusoids — trigonometric identities
4. **Attention patterns** — consistent with rotation composition

This notebook performs the equivalent analysis on every internal component of
the trained Label-JEPA model. If Fourier circuits exist anywhere in the network,
this audit will find them.

## Components to Audit

### Context Encoder
| Layer | Shape | What to check |
|-------|-------|---------------|
| `emb.weight` | (97, 256) | DFT along token axis — do embeddings lie on a circle? |
| `net.0` (Linear) | (512→256) | Hidden activations per residue class |
| `net.2` (Linear) | (256→256) | Hidden activations per residue class |
| `net.4` (Linear) | (256→128) | Pre-normalization output |

### Target Encoder (EMA)
| Layer | Shape | What to check |
|-------|-------|---------------|
| `emb.weight` | (97, 256) | DFT along token axis |
| `net.0` (Linear) | (256→256) | Hidden activations for residues 0..96 |
| `net.2` (Linear) | (256→128) | Pre-normalization output |

### Predictor
| Layer | Shape | What to check |
|-------|-------|---------------|
| `net.0` (Linear) | (128→64) | Weight matrix Fourier structure |
| `net.2` (Linear) | (64→64) | Bottleneck activations |
| `net.4` (Linear) | (64→128) | Output weight matrix |

## 1. Setup & Model Loading

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from copy import deepcopy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

p = 97
LATENT_DIM = 128
HIDDEN_DIM = 256
PREDICTOR_DIM = 64

## 2. Architecture (identical to core notebook)

In [ ]:
class ContextEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.size(0), -1)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class TargetEncoder(nn.Module):
    def __init__(self, vocab_size, latent_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x):
        e = self.emb(x)
        z = self.net(e)
        return F.normalize(z, dim=-1)


class Predictor(nn.Module):
    def __init__(self, latent_dim, predictor_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, predictor_dim),
            nn.GELU(),
            nn.Linear(predictor_dim, latent_dim),
        )

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)

print("Architecture defined.")

## 3. Load Trained Checkpoint

Load the checkpoint from the core notebook. If not available,
retrain with the same hyperparameters.

In [ ]:
import os
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
import time

# Generate data (same as core notebook)
pairs = torch.cartesian_prod(torch.arange(p), torch.arange(p))
targets = (pairs[:, 0] + pairs[:, 1]) % p
n = len(pairs)
n_train = int(0.3 * n)
perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:n_train], perm[n_train:]
train_pairs = pairs[train_idx].to(device)
train_targets = targets[train_idx].to(device)
val_pairs = pairs[val_idx].to(device)
val_targets = targets[val_idx].to(device)

# Try loading checkpoint
ckpt_path = "jepa_grokking_checkpoint.pt"
loaded = False

if os.path.exists(ckpt_path):
    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
        target_enc_ema = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
        predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
        context_enc.load_state_dict(ckpt["context_enc"])
        target_enc_ema.load_state_dict(ckpt["target_enc_ema"])
        predictor.load_state_dict(ckpt["predictor"])
        print(f"Loaded checkpoint from {ckpt_path}")
        loaded = True
    except Exception as e:
        print(f"Failed to load checkpoint: {e}")

if not loaded:
    print("No checkpoint found. Training from scratch...")
    print("(This will take ~20-50 minutes depending on hardware)")

    EPOCHS = 100_000
    LR = 1e-3
    WEIGHT_DECAY = 1.0
    EMA_DECAY = 0.996

    torch.manual_seed(SEED)
    context_enc = ContextEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    target_enc = TargetEncoder(p, LATENT_DIM, HIDDEN_DIM).to(device)
    predictor = Predictor(LATENT_DIM, PREDICTOR_DIM).to(device)
    target_enc_ema = deepcopy(target_enc)
    for param in target_enc_ema.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(
        list(context_enc.parameters()) +
        list(predictor.parameters()) +
        list(target_enc.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY,
    )

    train_loader = DataLoader(
        TensorDataset(train_pairs, train_targets),
        batch_size=n_train, shuffle=True,
    )

    start = time.time()
    for epoch in range(EPOCHS):
        context_enc.train(); target_enc.train(); predictor.train()
        for bp, bt in train_loader:
            optimizer.zero_grad()
            z_ctx = context_enc(bp)
            z_pred = predictor(z_ctx)
            with torch.no_grad():
                z_tgt = target_enc_ema(bt)
            loss = -(z_pred * z_tgt).sum(dim=-1).mean()
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                for o, t in zip(target_enc.parameters(), target_enc_ema.parameters()):
                    t.data.mul_(EMA_DECAY).add_(o.data, alpha=1 - EMA_DECAY)

        if epoch % 10000 == 0:
            context_enc.eval()
            with torch.no_grad():
                z_tr = context_enc(train_pairs)
                z_va = context_enc(val_pairs)
                y_tr = F.one_hot(train_targets, p).float()
                reg = 1e-3
                W = torch.linalg.solve(
                    z_tr.T @ z_tr + reg * torch.eye(LATENT_DIM, device=device),
                    z_tr.T @ y_tr,
                )
                val_acc = (z_va @ W).argmax(1).eq(val_targets).float().mean().item()
            print(f"  Epoch {epoch:6d} ({(time.time()-start)/60:.1f}m) | Val: {val_acc*100:.1f}%")

    torch.save({
        "context_enc": context_enc.state_dict(),
        "target_enc_ema": target_enc_ema.state_dict(),
        "predictor": predictor.state_dict(),
    }, ckpt_path)
    print(f"Training complete. Checkpoint saved.")

context_enc.eval()
target_enc_ema.eval()
predictor.eval()

# Quick validation
with torch.no_grad():
    z_tr = context_enc(train_pairs)
    z_va = context_enc(val_pairs)
    y_tr = F.one_hot(train_targets, p).float()
    W = torch.linalg.solve(
        z_tr.T @ z_tr + 1e-3 * torch.eye(LATENT_DIM, device=device),
        z_tr.T @ y_tr,
    )
    val_acc = (z_va @ W).argmax(1).eq(val_targets).float().mean().item()
print(f"\nModel validation accuracy: {val_acc*100:.1f}%")

## 4. Fourier Analysis Utilities

Reusable functions for computing Fourier structure in any matrix or activation tensor.

In [ ]:
def fourier_spectrum(matrix, axis=0):
    """Compute normalized Fourier power spectrum along given axis.

    Args:
        matrix: numpy array (p, D) or (D, p)
        axis: axis along which tokens are indexed (axis with length p)

    Returns:
        power: normalized power per frequency (length p)
        top5_ratio: fraction of power in top 5 frequencies (excluding DC)
    """
    fft = np.fft.fft(matrix, axis=axis)
    power_per_freq = (np.abs(fft) ** 2).sum(axis=1 - axis)  # sum over non-token axis
    power_per_freq[0] = 0  # remove DC
    total = power_per_freq.sum()
    if total < 1e-12:
        return power_per_freq, 0.0
    normalized = power_per_freq / total
    top5 = np.sort(power_per_freq)[-5:].sum() / total
    return normalized, top5


def plot_spectrum(ax, power, title, p, highlight_top=5):
    """Plot a Fourier spectrum bar chart."""
    half = p // 2 + 1
    ax.bar(range(half), power[:half], color="steelblue", alpha=0.8, width=1.0)
    ax.axhline(y=1/(p-1), color="red", linestyle="--", alpha=0.5, label=f"Uniform ({1/(p-1):.4f})")

    # Highlight top frequencies
    top_idx = np.argsort(power)[-highlight_top:]
    top_idx = top_idx[top_idx < half]
    for idx in top_idx:
        ax.bar(idx, power[idx], color="orange", alpha=0.9, width=1.0)

    top5_ratio = np.sort(power * (power.sum() if power.sum() > 0 else 1))[-5:].sum()
    ax.set_title(f"{title}\n(top-5 = {top5_ratio:.3f})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Frequency k")
    ax.set_ylabel("Normalized Power")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)


def check_circular_embedding(emb_weights, p, name=""):
    """Check if embedding weights have circular (Fourier) structure.

    Like Nanda's analysis of W_E: do tokens embed on a circle in any 2D subspace?
    """
    # PCA to find dominant structure
    pca = PCA(n_components=min(10, emb_weights.shape[1]))
    z_2d = pca.fit_transform(emb_weights)

    # Check if first two PCs form a circle (tokens ordered by index)
    angles = np.arctan2(z_2d[:, 1], z_2d[:, 0])
    # If circular, consecutive tokens should have roughly equal angular spacing
    angle_diffs = np.diff(np.sort(angles))
    angular_uniformity = angle_diffs.std() / (angle_diffs.mean() + 1e-10)

    # Fourier on the raw embeddings
    power, top5 = fourier_spectrum(emb_weights, axis=0)

    return {
        "pca_var_ratio": pca.explained_variance_ratio_,
        "z_2d": z_2d,
        "angles": angles,
        "angular_uniformity": angular_uniformity,
        "fourier_power": power,
        "fourier_top5": top5,
    }


print("Utilities defined.")

---
## Audit 1: Embedding Weights

Nanda's key finding: embedding matrix $W_E$ places tokens on a circle.
If Label-JEPA does the same, the tokens 0..96 should form circular structure
in the principal components of the embedding weights.

We check both the context encoder embedding and the target encoder (EMA) embedding.

In [ ]:
print("=" * 70)
print("AUDIT 1: EMBEDDING WEIGHTS")
print("=" * 70)

# Context encoder embedding
ctx_emb = context_enc.emb.weight.detach().cpu().numpy()  # (97, 256)
ctx_result = check_circular_embedding(ctx_emb, p, "Context Encoder")

# Target encoder embedding
tgt_emb = target_enc_ema.emb.weight.detach().cpu().numpy()  # (97, 256)
tgt_result = check_circular_embedding(tgt_emb, p, "Target Encoder")

print(f"\nContext Encoder Embedding (97 × 256):")
print(f"  Top-5 Fourier concentration: {ctx_result['fourier_top5']:.4f}")
print(f"  Uniform baseline: {5/p:.4f}")
print(f"  PCA var explained (PC1,PC2): {ctx_result['pca_var_ratio'][0]:.3f}, {ctx_result['pca_var_ratio'][1]:.3f}")
print(f"  Angular uniformity (lower=more circular): {ctx_result['angular_uniformity']:.3f}")

print(f"\nTarget Encoder (EMA) Embedding (97 × 256):")
print(f"  Top-5 Fourier concentration: {tgt_result['fourier_top5']:.4f}")
print(f"  Uniform baseline: {5/p:.4f}")
print(f"  PCA var explained (PC1,PC2): {tgt_result['pca_var_ratio'][0]:.3f}, {tgt_result['pca_var_ratio'][1]:.3f}")
print(f"  Angular uniformity (lower=more circular): {tgt_result['angular_uniformity']:.3f}")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Context encoder: spectrum, PCA, angular distribution
plot_spectrum(axes[0, 0], ctx_result["fourier_power"], "Context Enc: Embedding Spectrum", p)

ax = axes[0, 1]
sc = ax.scatter(ctx_result["z_2d"][:, 0], ctx_result["z_2d"][:, 1],
                c=np.arange(p), cmap="hsv", s=30, edgecolors="black", linewidth=0.3)
ax.set_title(f"Context Enc: Embedding PCA\n(var: {ctx_result['pca_var_ratio'][0]:.1%}, {ctx_result['pca_var_ratio'][1]:.1%})",
             fontsize=10, fontweight="bold")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.colorbar(sc, ax=ax, label="Token index")

ax = axes[0, 2]
ax.bar(range(min(20, len(ctx_result["pca_var_ratio"]))),
       ctx_result["pca_var_ratio"][:20] * 100,
       color="steelblue", alpha=0.8)
ax.set_title("Context Enc: PCA Variance Explained", fontsize=10, fontweight="bold")
ax.set_xlabel("Component"); ax.set_ylabel("Variance (%)")
ax.grid(True, alpha=0.3)

# Target encoder: spectrum, PCA, angular distribution
plot_spectrum(axes[1, 0], tgt_result["fourier_power"], "Target Enc (EMA): Embedding Spectrum", p)

ax = axes[1, 1]
sc = ax.scatter(tgt_result["z_2d"][:, 0], tgt_result["z_2d"][:, 1],
                c=np.arange(p), cmap="hsv", s=30, edgecolors="black", linewidth=0.3)
ax.set_title(f"Target Enc (EMA): Embedding PCA\n(var: {tgt_result['pca_var_ratio'][0]:.1%}, {tgt_result['pca_var_ratio'][1]:.1%})",
             fontsize=10, fontweight="bold")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.colorbar(sc, ax=ax, label="Token index")

ax = axes[1, 2]
ax.bar(range(min(20, len(tgt_result["pca_var_ratio"]))),
       tgt_result["pca_var_ratio"][:20] * 100,
       color="steelblue", alpha=0.8)
ax.set_title("Target Enc (EMA): PCA Variance Explained", fontsize=10, fontweight="bold")
ax.set_xlabel("Component"); ax.set_ylabel("Variance (%)")
ax.grid(True, alpha=0.3)

plt.suptitle("Audit 1: Do Embeddings Have Circular/Fourier Structure?",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit1_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 2: Hidden Layer Activations (Context Encoder)

For each hidden layer in the context encoder MLP, we:
1. Compute activations for all $97^2$ input pairs
2. Average per residue class $c = (a+b) \bmod p$
3. Apply DFT along the class axis

If Fourier circuits exist, the class-mean activations should show spectral concentration.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 2: HIDDEN LAYER ACTIVATIONS (Context Encoder)")
print("=" * 70)

context_enc.eval()
all_pairs_device = pairs.to(device)
targets_np = targets.numpy()

# Hook into each layer to capture activations
layer_activations = {}

def make_hook(name):
    def hook_fn(module, input, output):
        layer_activations[name] = output.detach()
    return hook_fn

# Register hooks on each layer in the context encoder MLP
hooks = []
hooks.append(context_enc.net[0].register_forward_hook(make_hook("linear1_pre_gelu")))
hooks.append(context_enc.net[1].register_forward_hook(make_hook("gelu1")))
hooks.append(context_enc.net[2].register_forward_hook(make_hook("linear2_pre_gelu")))
hooks.append(context_enc.net[3].register_forward_hook(make_hook("gelu2")))
hooks.append(context_enc.net[4].register_forward_hook(make_hook("linear3_output")))

# Also capture embedding output
def emb_hook(module, input, output):
    layer_activations["embedding"] = output.detach()
hooks.append(context_enc.emb.register_forward_hook(emb_hook))

# Forward pass through all pairs
with torch.no_grad():
    _ = context_enc(all_pairs_device)

# Remove hooks
for h in hooks:
    h.remove()

# Compute per-class mean activations and Fourier structure for each layer
layer_fourier = {}

for layer_name, acts in layer_activations.items():
    acts_np = acts.cpu().numpy()
    if acts_np.ndim == 3:
        # Embedding: (9409, 2, 256) → flatten to (9409, 512)
        acts_np = acts_np.reshape(acts_np.shape[0], -1)

    # Average per residue class
    class_means = np.zeros((p, acts_np.shape[1]))
    for c in range(p):
        mask = targets_np == c
        class_means[c] = acts_np[mask].mean(axis=0)

    # Fourier analysis
    power, top5 = fourier_spectrum(class_means, axis=0)
    layer_fourier[layer_name] = {
        "class_means": class_means,
        "power": power,
        "top5": top5,
        "shape": acts_np.shape,
    }

    print(f"  {layer_name:25s} | shape: {str(acts_np.shape):15s} | Fourier top-5: {top5:.4f}")

# Visualization
layer_names = list(layer_fourier.keys())
n_layers = len(layer_names)
fig, axes = plt.subplots(2, (n_layers + 1) // 2, figsize=(5 * ((n_layers + 1) // 2), 10))
axes = axes.flatten()

for i, name in enumerate(layer_names):
    plot_spectrum(axes[i], layer_fourier[name]["power"],
                  f"Ctx {name}\n(top5={layer_fourier[name]['top5']:.3f})", p)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Audit 2: Context Encoder Hidden Layer Fourier Spectra\n(per-class-mean activations)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit2_ctx_hidden.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 3: Hidden Layer Activations (Target Encoder EMA)

Same analysis for the target encoder. Since it only takes single tokens (0..96),
we directly pass each residue and analyze the per-layer activations.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 3: HIDDEN LAYER ACTIVATIONS (Target Encoder EMA)")
print("=" * 70)

tgt_layer_activations = {}

def make_tgt_hook(name):
    def hook_fn(module, input, output):
        tgt_layer_activations[name] = output.detach()
    return hook_fn

tgt_hooks = []
tgt_hooks.append(target_enc_ema.emb.register_forward_hook(make_tgt_hook("tgt_embedding")))
tgt_hooks.append(target_enc_ema.net[0].register_forward_hook(make_tgt_hook("tgt_linear1")))
tgt_hooks.append(target_enc_ema.net[1].register_forward_hook(make_tgt_hook("tgt_gelu")))
tgt_hooks.append(target_enc_ema.net[2].register_forward_hook(make_tgt_hook("tgt_output")))

with torch.no_grad():
    _ = target_enc_ema(torch.arange(p, device=device))

for h in tgt_hooks:
    h.remove()

tgt_layer_fourier = {}
for layer_name, acts in tgt_layer_activations.items():
    acts_np = acts.cpu().numpy()  # (97, D) — already one per residue
    power, top5 = fourier_spectrum(acts_np, axis=0)
    tgt_layer_fourier[layer_name] = {
        "activations": acts_np,
        "power": power,
        "top5": top5,
    }
    print(f"  {layer_name:20s} | shape: {str(acts_np.shape):12s} | Fourier top-5: {top5:.4f}")

# Visualization
tgt_names = list(tgt_layer_fourier.keys())
fig, axes = plt.subplots(1, len(tgt_names), figsize=(5 * len(tgt_names), 5))
if len(tgt_names) == 1:
    axes = [axes]

for i, name in enumerate(tgt_names):
    plot_spectrum(axes[i], tgt_layer_fourier[name]["power"],
                  f"TgtEMA {name}\n(top5={tgt_layer_fourier[name]['top5']:.3f})", p)

plt.suptitle("Audit 3: Target Encoder (EMA) Hidden Layer Fourier Spectra",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit3_tgt_hidden.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 4: Predictor Activations

The predictor maps context latents to predicted target latents.
We check whether its internal bottleneck activations show Fourier structure
when averaged per residue class.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 4: PREDICTOR ACTIVATIONS")
print("=" * 70)

pred_layer_activations = {}

def make_pred_hook(name):
    def hook_fn(module, input, output):
        pred_layer_activations[name] = output.detach()
    return hook_fn

pred_hooks = []
pred_hooks.append(predictor.net[0].register_forward_hook(make_pred_hook("pred_linear1")))
pred_hooks.append(predictor.net[1].register_forward_hook(make_pred_hook("pred_gelu1")))
pred_hooks.append(predictor.net[2].register_forward_hook(make_pred_hook("pred_bottleneck")))
pred_hooks.append(predictor.net[3].register_forward_hook(make_pred_hook("pred_gelu2")))
pred_hooks.append(predictor.net[4].register_forward_hook(make_pred_hook("pred_output")))

with torch.no_grad():
    z_ctx_all = context_enc(all_pairs_device)
    _ = predictor(z_ctx_all)

for h in pred_hooks:
    h.remove()

pred_layer_fourier = {}
for layer_name, acts in pred_layer_activations.items():
    acts_np = acts.cpu().numpy()

    # Average per residue class
    class_means = np.zeros((p, acts_np.shape[1]))
    for c in range(p):
        mask = targets_np == c
        class_means[c] = acts_np[mask].mean(axis=0)

    power, top5 = fourier_spectrum(class_means, axis=0)
    pred_layer_fourier[layer_name] = {
        "class_means": class_means,
        "power": power,
        "top5": top5,
    }
    print(f"  {layer_name:20s} | shape: {str(acts_np.shape):15s} | Fourier top-5: {top5:.4f}")

# Visualization
pred_names = list(pred_layer_fourier.keys())
fig, axes = plt.subplots(1, len(pred_names), figsize=(5 * len(pred_names), 5))

for i, name in enumerate(pred_names):
    plot_spectrum(axes[i], pred_layer_fourier[name]["power"],
                  f"Pred {name}\n(top5={pred_layer_fourier[name]['top5']:.3f})", p)

plt.suptitle("Audit 4: Predictor Hidden Layer Fourier Spectra\n(per-class-mean activations)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("audit4_predictor.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Audit 5: Individual Neuron Response Patterns

Nanda found that individual MLP neurons responded as $\cos(\omega_k(a+b))$ or
$\sin(\omega_k(a+b))$ for specific key frequencies $k$. We check whether any
individual neurons in the context encoder MLP show sinusoidal response patterns.

For each neuron, we:
1. Compute its activation for all $97^2$ inputs
2. Average by residue class → get neuron's response as function of $c$
3. Compute the Fourier concentration of that single neuron
4. Check if any neuron has dominant single frequency (sinusoidal response)

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 5: INDIVIDUAL NEURON SINUSOIDAL RESPONSES")
print("=" * 70)

# Use the hidden activations already captured
# Focus on GELU outputs (post-activation) — these are what Nanda analyzed

for layer_name in ["gelu1", "gelu2"]:
    data = layer_fourier[layer_name]
    class_means = data["class_means"]  # (97, D)
    n_neurons = class_means.shape[1]

    # For each neuron, compute Fourier concentration
    neuron_top1 = []
    neuron_top1_freq = []
    neuron_top3 = []

    for d in range(n_neurons):
        neuron_signal = class_means[:, d]
        fft = np.fft.fft(neuron_signal)
        power = np.abs(fft) ** 2
        power[0] = 0  # remove DC
        total = power.sum()
        if total < 1e-12:
            neuron_top1.append(0)
            neuron_top1_freq.append(0)
            neuron_top3.append(0)
            continue

        sorted_power = np.sort(power)
        neuron_top1.append(sorted_power[-1] / total)
        neuron_top1_freq.append(np.argmax(power))
        neuron_top3.append(sorted_power[-3:].sum() / total)

    neuron_top1 = np.array(neuron_top1)
    neuron_top3 = np.array(neuron_top3)
    neuron_top1_freq = np.array(neuron_top1_freq)

    print(f"\n  {layer_name} ({n_neurons} neurons):")
    print(f"    Max single-freq concentration: {neuron_top1.max():.4f} (neuron {neuron_top1.argmax()}, freq={neuron_top1_freq[neuron_top1.argmax()]})")
    print(f"    Mean single-freq concentration: {neuron_top1.mean():.4f}")
    print(f"    Neurons with top1 > 0.3: {(neuron_top1 > 0.3).sum()}/{n_neurons}")
    print(f"    Neurons with top1 > 0.5: {(neuron_top1 > 0.5).sum()}/{n_neurons}")
    print(f"    Neurons with top1 > 0.7: {(neuron_top1 > 0.7).sum()}/{n_neurons}")

    # For comparison: in Nanda's model, key neurons have top1 > 0.8
    # If our model is non-Fourier, we expect no neurons with high concentration

    # Plot distribution and show top neurons
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Distribution of neuron Fourier concentrations
    ax = axes[0]
    ax.hist(neuron_top1, bins=50, color="steelblue", alpha=0.8, edgecolor="black", linewidth=0.5)
    ax.axvline(x=0.3, color="red", linestyle="--", alpha=0.5, label="0.3 threshold")
    ax.axvline(x=0.5, color="red", linestyle="-", alpha=0.5, label="0.5 threshold")
    ax.set_title(f"{layer_name}: Top-1 Frequency Concentration\nper Neuron", fontsize=11, fontweight="bold")
    ax.set_xlabel("Top-1 Freq / Total Power")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Which frequencies are dominant across neurons
    ax = axes[1]
    freq_counts = np.bincount(neuron_top1_freq[neuron_top1 > 0.1], minlength=p)
    ax.bar(range(p // 2 + 1), freq_counts[:p // 2 + 1], color="steelblue", alpha=0.8)
    ax.set_title(f"{layer_name}: Dominant Frequency Distribution\n(neurons with top1 > 0.1)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Frequency k")
    ax.set_ylabel("# Neurons")
    ax.grid(True, alpha=0.3)

    # Show top 5 most sinusoidal neurons
    ax = axes[2]
    top_neurons = np.argsort(neuron_top1)[-5:][::-1]
    for rank, n_idx in enumerate(top_neurons):
        signal = class_means[:, n_idx]
        signal_norm = (signal - signal.mean()) / (signal.std() + 1e-10)
        ax.plot(signal_norm, label=f"Neuron {n_idx} (top1={neuron_top1[n_idx]:.2f}, k={neuron_top1_freq[n_idx]})",
                alpha=0.8)
    ax.set_title(f"{layer_name}: Top 5 Most Sinusoidal Neurons\n(class-mean response)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Residue class c")
    ax.set_ylabel("Normalized activation")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"audit5_{layer_name}_neurons.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## Audit 6: Weight Matrix Fourier Structure

Nanda analyzed the neuron-logit map $W_L$ (the last weight matrix) and found it
was well-approximated by sinusoidal functions of key frequencies. We do the
equivalent analysis on all weight matrices in the context encoder and predictor.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 6: WEIGHT MATRIX FOURIER STRUCTURE")
print("=" * 70)

# Collect all weight matrices
weight_matrices = {}

# Context encoder
weight_matrices["ctx_emb"] = context_enc.emb.weight.detach().cpu().numpy()  # (97, 256)
weight_matrices["ctx_linear1_W"] = context_enc.net[0].weight.detach().cpu().numpy()  # (256, 512)
weight_matrices["ctx_linear2_W"] = context_enc.net[2].weight.detach().cpu().numpy()  # (256, 256)
weight_matrices["ctx_linear3_W"] = context_enc.net[4].weight.detach().cpu().numpy()  # (128, 256)

# Target encoder (EMA)
weight_matrices["tgt_emb"] = target_enc_ema.emb.weight.detach().cpu().numpy()  # (97, 256)
weight_matrices["tgt_linear1_W"] = target_enc_ema.net[0].weight.detach().cpu().numpy()  # (256, 256)
weight_matrices["tgt_linear2_W"] = target_enc_ema.net[2].weight.detach().cpu().numpy()  # (128, 256)

# Predictor
weight_matrices["pred_down_W"] = predictor.net[0].weight.detach().cpu().numpy()  # (64, 128)
weight_matrices["pred_mid_W"] = predictor.net[2].weight.detach().cpu().numpy()  # (64, 64)
weight_matrices["pred_up_W"] = predictor.net[4].weight.detach().cpu().numpy()  # (128, 64)

for name, W in weight_matrices.items():
    # SVD to find dominant structure
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    effective_rank = (S ** 2).sum() ** 2 / (S ** 4).sum()
    top1_ratio = S[0] ** 2 / (S ** 2).sum()

    # For embedding matrices (97 × D), check Fourier along token axis
    if W.shape[0] == p:
        power, top5 = fourier_spectrum(W, axis=0)
        print(f"  {name:20s} | shape: {str(W.shape):12s} | eff_rank: {effective_rank:.1f} | top1_sv: {top1_ratio:.3f} | Fourier top5: {top5:.4f}")
    else:
        # For non-embedding matrices, check Fourier along both axes
        power_r, top5_r = fourier_spectrum(W, axis=0)
        power_c, top5_c = fourier_spectrum(W, axis=1)
        print(f"  {name:20s} | shape: {str(W.shape):12s} | eff_rank: {effective_rank:.1f} | top1_sv: {top1_ratio:.3f} | Fourier rows: {top5_r:.4f}, cols: {top5_c:.4f}")

---
## Audit 7: Nanda-Style Complete Circuit Analysis

Nanda's full analysis: project MLP activations onto $\cos(2\pi k \cdot / p)$ and
$\sin(2\pi k \cdot / p)$ for each frequency $k$, and check whether activations
combine as trigonometric identities like $\cos(\omega(a+b)) = \cos(\omega a)\cos(\omega b) - \sin(\omega a)\sin(\omega b)$.

If the model implements addition via rotation, this projection should yield high $R^2$.
If not, it should be near chance.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 7: TRIGONOMETRIC IDENTITY TEST (Nanda Circuit Analysis)")
print("=" * 70)

# Build Fourier features for inputs a and b
a_vals = pairs[:, 0].numpy()
b_vals = pairs[:, 1].numpy()

freqs_to_test = list(range(1, p // 2 + 1))  # all non-DC frequencies

# For each hidden layer, test: can activations be predicted by
# cos(2πk·a/p), sin(2πk·a/p), cos(2πk·b/p), sin(2πk·b/p) and their products?

from sklearn.linear_model import Ridge

for layer_name in ["gelu1", "gelu2", "linear3_output"]:
    acts = layer_activations[layer_name].cpu().numpy()
    if acts.ndim == 3:
        acts = acts.reshape(acts.shape[0], -1)

    # Test each frequency: how well do trig features of (a,b) at freq k predict activations?
    freq_r2 = []

    for k in freqs_to_test:
        omega = 2 * np.pi * k / p
        # 4 features: cos(ωa), sin(ωa), cos(ωb), sin(ωb)
        # Plus 4 product features: cos(ωa)cos(ωb), cos(ωa)sin(ωb), sin(ωa)cos(ωb), sin(ωa)sin(ωb)
        cos_a = np.cos(omega * a_vals)
        sin_a = np.sin(omega * a_vals)
        cos_b = np.cos(omega * b_vals)
        sin_b = np.sin(omega * b_vals)

        X = np.column_stack([
            cos_a, sin_a, cos_b, sin_b,
            cos_a * cos_b, cos_a * sin_b,
            sin_a * cos_b, sin_a * sin_b,
        ])

        # Fit ridge regression to predict all activations from these trig features
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)

        # R² per neuron, then average
        ss_res = ((acts - y_pred) ** 2).sum(axis=0)
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum(axis=0)
        r2_per_neuron = 1 - ss_res / (ss_tot + 1e-10)
        r2_mean = r2_per_neuron.mean()
        freq_r2.append(r2_mean)

    freq_r2 = np.array(freq_r2)
    best_freq = freqs_to_test[np.argmax(freq_r2)]
    best_r2 = freq_r2.max()
    mean_r2 = freq_r2.mean()

    print(f"\n  {layer_name}:")
    print(f"    Best single-frequency R²: {best_r2:.4f} (k={best_freq})")
    print(f"    Mean R² across all frequencies: {mean_r2:.4f}")
    print(f"    Frequencies with R² > 0.3: {(freq_r2 > 0.3).sum()}/{len(freqs_to_test)}")
    print(f"    Frequencies with R² > 0.5: {(freq_r2 > 0.5).sum()}/{len(freqs_to_test)}")

    # For Nanda's model, key frequencies had R² > 0.8
    # For non-Fourier model, we expect all R² near 0

---
## Audit 7b: Multi-Frequency Combined R²

Even if no single frequency dominates, perhaps several frequencies together
explain the activations (like Nanda's 5 key frequencies). We test progressively
adding frequencies by their individual R² and check combined explanatory power.

In [ ]:
print("\n" + "=" * 70)
print("AUDIT 7b: MULTI-FREQUENCY COMBINED R²")
print("=" * 70)

for layer_name in ["gelu2", "linear3_output"]:
    acts = layer_activations[layer_name].cpu().numpy()
    if acts.ndim == 3:
        acts = acts.reshape(acts.shape[0], -1)

    # Compute single-frequency R² for ranking
    single_r2 = []
    for k in freqs_to_test:
        omega = 2 * np.pi * k / p
        cos_a = np.cos(omega * a_vals)
        sin_a = np.sin(omega * a_vals)
        cos_b = np.cos(omega * b_vals)
        sin_b = np.sin(omega * b_vals)
        X = np.column_stack([cos_a, sin_a, cos_b, sin_b,
                             cos_a*cos_b, cos_a*sin_b, sin_a*cos_b, sin_a*sin_b])
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)
        ss_res = ((acts - y_pred) ** 2).sum()
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum()
        single_r2.append(1 - ss_res / (ss_tot + 1e-10))
    single_r2 = np.array(single_r2)

    # Rank frequencies by individual R²
    freq_order = np.argsort(single_r2)[::-1]

    # Incrementally add frequencies and measure combined R²
    combined_r2 = []
    for n_freqs in range(1, min(21, len(freqs_to_test) + 1)):
        selected = freq_order[:n_freqs]
        X_parts = []
        for idx in selected:
            k = freqs_to_test[idx]
            omega = 2 * np.pi * k / p
            cos_a = np.cos(omega * a_vals)
            sin_a = np.sin(omega * a_vals)
            cos_b = np.cos(omega * b_vals)
            sin_b = np.sin(omega * b_vals)
            X_parts.extend([cos_a, sin_a, cos_b, sin_b,
                            cos_a*cos_b, cos_a*sin_b, sin_a*cos_b, sin_a*sin_b])
        X = np.column_stack(X_parts)
        ridge = Ridge(alpha=1.0)
        ridge.fit(X, acts)
        y_pred = ridge.predict(X)
        ss_res = ((acts - y_pred) ** 2).sum()
        ss_tot = ((acts - acts.mean(axis=0)) ** 2).sum()
        combined_r2.append(1 - ss_res / (ss_tot + 1e-10))

    print(f"\n  {layer_name}:")
    for n in [1, 3, 5, 10, 20]:
        if n <= len(combined_r2):
            top_freqs = [freqs_to_test[freq_order[i]] for i in range(n)]
            print(f"    Top {n:2d} freqs: R² = {combined_r2[n-1]:.4f}  (freqs: {top_freqs[:5]}{'...' if n > 5 else ''})")

    # For Nanda: 5 key frequencies → R² > 0.95
    # For non-Fourier: even 20 frequencies should give low R²

---
## Audit 8: Summary Dashboard

In [ ]:
print("\n" + "=" * 70)
print("COMPLETE FOURIER AUDIT SUMMARY")
print("=" * 70)

print("\n--- Embedding Weights ---")
print(f"  Context encoder embedding: Fourier top-5 = {ctx_result['fourier_top5']:.4f} (uniform: {5/p:.4f})")
print(f"  Target encoder embedding:  Fourier top-5 = {tgt_result['fourier_top5']:.4f} (uniform: {5/p:.4f})")

print("\n--- Hidden Layer Activations (per-class mean) ---")
for name, data in layer_fourier.items():
    print(f"  Context {name:25s}: Fourier top-5 = {data['top5']:.4f}")
for name, data in tgt_layer_fourier.items():
    print(f"  Target  {name:25s}: Fourier top-5 = {data['top5']:.4f}")

print("\n--- Predictor Activations (per-class mean) ---")
for name, data in pred_layer_fourier.items():
    print(f"  {name:25s}: Fourier top-5 = {data['top5']:.4f}")

print("\n--- Individual Neuron Sinusoidal Responses ---")
for layer_name in ["gelu1", "gelu2"]:
    data = layer_fourier[layer_name]
    class_means = data["class_means"]
    n_neurons = class_means.shape[1]
    neuron_top1 = []
    for d in range(n_neurons):
        fft = np.fft.fft(class_means[:, d])
        power = np.abs(fft) ** 2
        power[0] = 0
        total = power.sum()
        neuron_top1.append(power.max() / total if total > 1e-12 else 0)
    neuron_top1 = np.array(neuron_top1)
    print(f"  {layer_name}: max neuron top-1 = {neuron_top1.max():.4f}, "
          f"neurons > 0.5 = {(neuron_top1 > 0.5).sum()}/{n_neurons}")

# Verdict
print("\n" + "=" * 70)

all_top5 = []
for data in layer_fourier.values():
    all_top5.append(data["top5"])
for data in tgt_layer_fourier.values():
    all_top5.append(data["top5"])
for data in pred_layer_fourier.values():
    all_top5.append(data["top5"])
all_top5.extend([ctx_result["fourier_top5"], tgt_result["fourier_top5"]])

max_fourier = max(all_top5)
uniform = 5 / p

if max_fourier > 0.5:
    print(f"⚠  FOURIER STRUCTURE DETECTED (max top-5 = {max_fourier:.4f})")
    print(f"   The 'no Fourier circuits' claim needs qualification.")
elif max_fourier > 0.2:
    print(f"⚡ MILD FOURIER CONCENTRATION (max top-5 = {max_fourier:.4f})")
    print(f"   Some spectral structure exists but is not dominant.")
    print(f"   Compare: uniform baseline = {uniform:.4f}")
else:
    print(f"✓  NO FOURIER STRUCTURE FOUND (max top-5 = {max_fourier:.4f})")
    print(f"   Across all layers, embeddings, and individual neurons.")
    print(f"   Uniform baseline = {uniform:.4f}")

print("=" * 70)